In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import concurrent.futures
import functools

from func_file_Simulate import Get_PSF_kernel
from func_file_Process import *

In [ ]:
#Create folder for storing generated data files
if not os.path.exists("Data_files"):
    os.mkdir("Data_files")
if not os.path.exists("Data_files/Processed_data"):
    os.mkdir("Data_files/Processed_data")

# Load the ConvNet model and Specify parameters for Richardson-Lucy algorithms

In [ ]:
#Load the ConvNet model
model_ConvNet = Load_ConvNet_model("DAMN_ConvNet.h5")

In [ ]:
#Specify the PSF_width value used to generate data for panels (A), (C), and (D); requirement for Richardson-Lucy algorithm
PSF_width_value = 2
kernel = Gauss_kernel(PSF_width_value)   #The kernel stays the same for SNR, Concentration, and Transition data samples

#To fasten the Richardson-Lucy algorithm, we split the data evaluation among several CPU units using ProcessPoolExecutor from concurrent.futures
CPU_units_to_use = 4

# Evaluate dataset of the SNR graph - panel (A)

In [ ]:
#Loading the data generated in the Data_generation notebook
data_in_SNR = np.load("Data_files/Generated_data/SNR_data_low_res.npy")
data_target_SNR = np.load("Data_files/Generated_data/SNR_data_high_res.npy")

shape_SNR = data_in_SNR.shape
print("Data shapes:", shape_SNR)

In [ ]:
#As the RL algorithm is very time consuming, you can choose to evaluate only a small portion of the data for the graph visualization
#Namely, 1/10 of data samples in every 4-th point on the horizontal axis
use_all_data_SNR = False             #Switch to False for evaluating only the reduced dataset 

if use_all_data_SNR:
    RL_data_in_SNR = data_in_SNR
    RL_data_target_SNR = data_target_SNR
    
    RL_output_SNR = np.zeros(data_in_SNR.shape)
    RLTV_output_SNR = np.zeros(data_in_SNR.shape)
    
else:
    RL_data_in_SNR = data_in_SNR[::4,:10]
    RL_data_target_SNR = data_target_SNR[::4,:10]
    
    RL_output_SNR = np.zeros(data_in_SNR[::4,:10].shape)
    RL_TV_output_SNR = np.zeros(data_in_SNR[::4,:10].shape)

print("Using the following data shape for Richardson-Lucy algorithms:", RL_data_in_SNR.shape)

## The DAMN model

In [ ]:
#Use the DAMN model to predict high-resolution images
DAMN_model_output_SNR = np.reshape(model_ConvNet.predict(data_in_SNR.reshape([shape_SNR[0]*shape_SNR[1], shape_SNR[2], shape_SNR[3]]), 
                                                         verbose = 1), shape_SNR)

#Evaluate a chosen metric comparing with corresponding target images
DAMN_model_errors_SNR = Evaluate_metric(DAMN_model_output_SNR, data_target_SNR)

np.save("Data_files/Processed_data/SNR_DAMN_model_output.npy", DAMN_model_output_SNR)
np.save("Data_files/Processed_data/SNR_DAMN_model_errors.npy", DAMN_model_errors_SNR)

## Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_iteration_for_concurrent function to evaluate the RL_data_in_SNR data
start = time.time()
for i in range(RL_data_in_SNR.shape[0]):
    start_i = time.time()
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_iteration_for_concurrent, kernel)
        res = pool.map(intermediate_func, RL_data_in_SNR[i])
    RL_output_SNR[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_SNR.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/SNR_RL_output.npy", RL_output_SNR)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_errors_SNR = Evaluate_metric(RL_output_SNR, RL_data_target_SNR)

np.save("Data_files/Processed_data/SNR_RL_errors.npy", RL_errors_SNR)

## Total variation Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_TV_iteration_for_concurrent function to evaluate the RL_data_in_SNR data
start = time.time()
for i in range(RL_data_in_SNR.shape[0]):
    start_i = time.time()
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_TV_iteration_for_concurrent, kernel)
        res = pool.map(intermediate_func, RL_data_in_SNR[i])
    RL_TV_output_SNR[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_SNR.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/SNR_RLTV_output.npy", RL_TV_output_SNR)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_TV_errors_SNR = Evaluate_metric(RL_TV_output_SNR, RL_data_target_SNR)

np.save("Data_files/Processed_data/SNR_RLTV_errors.npy", RL_TV_errors_SNR)

## Evaluate dataset of the PSF graph - panel (B)

In [ ]:
#Loading the data generated in the Data_generation notebook
data_in_PSF = np.load("Data_files/Generated_data/PSF_data_low_res.npy")
data_target_PSF = np.load("Data_files/Generated_data/PSF_data_high_res.npy")
horizontal_axis_PSF = np.load("Data_files/Generated_data/PSF_axis_array.npy")

shape_PSF = data_in_PSF.shape
print("Data shapes:", shape_PSF)

In [ ]:
#As the RL algorithm is very time consuming, you can choose to evaluate only a small portion of the data for the graph visualization
#Namely, 1/10 of data samples in every 4-th point on the horizontal axis
use_all_data_PSF = False             #Switch to False for evaluating only the reduced dataset 

if use_all_data_PSF:
    RL_data_in_PSF = data_in_PSF
    RL_data_target_PSF = data_target_PSF
    RL_horizontal_axis_PSF = horizontal_axis_PSF
    
    RL_output_PSF = np.zeros(data_in_PSF.shape)
    RLTV_output_PSF = np.zeros(data_in_PSF.shape)
    
else:
    RL_data_in_PSF = data_in_PSF[::4,:10]
    RL_data_target_PSF = data_target_PSF[::4,:10]
    RL_horizontal_axis_PSF = horizontal_axis_PSF[::4]
    
    RL_output_PSF = np.zeros(data_in_PSF[::4,:10].shape)
    RL_TV_output_PSF = np.zeros(data_in_PSF[::4,:10].shape)

print("Using the following data shape for Richardson-Lucy algorithms:", RL_data_in_PSF.shape)

## The DAMN model

In [ ]:
#Use the DAMN model to predict high-resolution images
DAMN_model_output_PSF = np.reshape(model_ConvNet.predict(data_in_PSF.reshape([shape_PSF[0]*shape_PSF[1], shape_PSF[2], shape_PSF[3]]), 
                                                         verbose = 1), shape_PSF)

#Evaluate a chosen metric comparing with corresponding target images
DAMN_model_errors_PSF = Evaluate_metric(DAMN_model_output_PSF, data_target_PSF)

np.save("Data_files/Processed_data/PSF_DAMN_model_output.npy", DAMN_model_output_PSF)
np.save("Data_files/Processed_data/PSF_DAMN_model_errors.npy", DAMN_model_errors_PSF)

## Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_iteration_for_concurrent function to evaluate the RL_data_in_PSF data
start = time.time()
for i in range(RL_data_in_PSF.shape[0]):
    start_i = time.time()
    
    #The kernel stays the same only for data with the same horizontal axis position
    kernel_PSF = Gauss_kernel(RL_horizontal_axis_PSF[i])
    
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_iteration_for_concurrent, kernel_PSF)
        res = pool.map(intermediate_func, RL_data_in_PSF[i])
    RL_output_PSF[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_PSF.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/PSF_RL_output.npy", RL_output_PSF)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_errors_PSF = Evaluate_metric(RL_output_PSF, RL_data_target_PSF)

np.save("Data_files/Processed_data/PSF_RL_errors.npy", RL_errors_PSF)

## Total variation Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_TV_iteration_for_concurrent function to evaluate the RL_data_in_PSF data
start = time.time()
for i in range(RL_data_in_PSF.shape[0]):
    start_i = time.time()
    
    #The kernel stays the same only for data with the same horizontal axis position
    kernel_PSF = Gauss_kernel(RL_horizontal_axis_PSF[i])
    
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_TV_iteration_for_concurrent, kernel_PSF)
        res = pool.map(intermediate_func, RL_data_in_PSF[i])
    RL_TV_output_PSF[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_PSF.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/PSF_RLTV_output.npy", RL_TV_output_PSF)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_TV_errors_PSF = Evaluate_metric(RL_TV_output_PSF, RL_data_target_PSF)

np.save("Data_files/Processed_data/PSF_RLTV_errors.npy", RL_TV_errors_PSF)

# Evaluate dataset of the Emitter Concentration graph - panel (C)

In [ ]:
#Loading the data generated in the Data_generation notebook
data_in_Conc = np.load("Data_files/Generated_data/Concentration_data_low_res.npy")
data_target_Conc = np.load("Data_files/Generated_data/Concentration_data_high_res.npy")

shape_Conc = data_in_Conc.shape
print("Data shapes:", shape_Conc)

In [ ]:
#As the RL algorithm is very time consuming, you can choose to evaluate only a small portion of the data for the graph visualization
#Namely, 1/10 of data samples in every 4-th point on the horizontal axis
use_all_data_Conc = False             #Switch to False for evaluating only the reduced dataset 

if use_all_data_Conc:
    RL_data_in_Conc = data_in_Conc
    RL_data_target_Conc = data_target_Conc
    
    RL_output_Conc = np.zeros(data_in_Conc.shape)
    RLTV_output_Conc = np.zeros(data_in_Conc.shape)
    
else:
    RL_data_in_Conc = data_in_Conc[::4,:10]
    RL_data_target_Conc = data_target_Conc[::4,:10]
    
    RL_output_Conc = np.zeros(data_in_Conc[::4,:10].shape)
    RL_TV_output_Conc = np.zeros(data_in_Conc[::4,:10].shape)

print("Using the following data shape for Richardson-Lucy algorithms:", RL_data_in_Conc.shape)

## The DAMN model

In [ ]:
#Use the DAMN model to predict high-resolution images
DAMN_model_output_Conc = np.reshape(model_ConvNet.predict(data_in_Conc.reshape([shape_Conc[0]*shape_Conc[1], shape_Conc[2], shape_Conc[3]]), 
                                                         verbose = 1), shape_Conc)

#Evaluate a chosen metric comparing with corresponding target images
DAMN_model_errors_Conc = Evaluate_metric(DAMN_model_output_Conc, data_target_Conc)

np.save("Data_files/Processed_data/Concentration_DAMN_model_output.npy", DAMN_model_output_Conc)
np.save("Data_files/Processed_data/Concentration_DAMN_model_errors.npy", DAMN_model_errors_Conc)

## Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_iteration_for_concurrent function to evaluate the RL_data_in_Conc data
start = time.time()
for i in range(RL_data_in_Conc.shape[0]):
    start_i = time.time()
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_iteration_for_concurrent, kernel)
        res = pool.map(intermediate_func, RL_data_in_Conc[i])
    RL_output_Conc[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_Conc.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/Concentration_RL_output.npy", RL_output_Conc)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_errors_Conc = Evaluate_metric(RL_output_Conc, RL_data_target_Conc)

np.save("Data_files/Processed_data/Concentration_RL_errors.npy", RL_errors_Conc)

## Total variation Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_TV_iteration_for_concurrent function to evaluate the RL_data_in_Conc data
start = time.time()
for i in range(RL_data_in_Conc.shape[0]):
    start_i = time.time()
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_TV_iteration_for_concurrent, kernel)
        res = pool.map(intermediate_func, RL_data_in_Conc[i])
    RL_TV_output_Conc[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_Conc.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/Concentration_RLTV_output.npy", RL_TV_output_Conc)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_TV_errors_Conc = Evaluate_metric(RL_TV_output_Conc, RL_data_target_Conc)

np.save("Data_files/Processed_data/Concentration_RLTV_errors.npy", RL_TV_errors_Conc)

# Evaluate dataset of the Gauss-to-Airy kernel transition graph - panel (D)

In [ ]:
#Loading the data generated in the Data_generation notebook
data_in_GtoA = np.load("Data_files/Generated_data/Transition_data_low_res.npy")
data_target_GtoA = np.load("Data_files/Generated_data/Transition_data_high_res.npy")
horizontal_axis_GtoA = np.load("Data_files/Generated_data/Transition_axis_array.npy")

shape_GtoA = data_in_GtoA.shape
print("Data shapes:", shape_GtoA)

In [ ]:
#As the RL algorithm is very time consuming, you can choose to evaluate only a small portion of the data for the graph visualization
#Namely, 1/10 of data samples in every 4-th point on the horizontal axis
use_all_data_GtoA = False             #Switch to False for evaluating only the reduced dataset 

if use_all_data_GtoA:
    RL_data_in_GtoA = data_in_GtoA
    RL_data_target_GtoA = data_target_GtoA
    RL_horizontal_axis_GtoA = horizontal_axis_GtoA
    
    RL_output_GtoA = np.zeros(data_in_GtoA.shape)
    RLTV_output_GtoA = np.zeros(data_in_GtoA.shape)
    
else:
    RL_data_in_GtoA = data_in_GtoA[::4,:10]
    RL_data_target_GtoA = data_target_GtoA[::4,:10]
    RL_horizontal_axis_GtoA = horizontal_axis_GtoA[::4]
    
    RL_output_GtoA = np.zeros(data_in_GtoA[::4,:10].shape)
    RL_TV_output_GtoA = np.zeros(data_in_GtoA[::4,:10].shape)

print("Using the following data shape for Richardson-Lucy algorithms:", RL_data_in_GtoA.shape)

## The DAMN model

In [ ]:
#Use the DAMN model to predict high-resolution images
DAMN_model_output_GtoA = np.reshape(model_ConvNet.predict(data_in_GtoA.reshape([shape_GtoA[0]*shape_GtoA[1], shape_GtoA[2], shape_GtoA[3]]), 
                                                         verbose = 1), shape_GtoA)

#Evaluate a chosen metric comparing with corresponding target images
DAMN_model_errors_GtoA = Evaluate_metric(DAMN_model_output_GtoA, data_target_GtoA)

np.save("Data_files/Processed_data/Transition_DAMN_model_output.npy", DAMN_model_output_GtoA)
np.save("Data_files/Processed_data/Transition_DAMN_model_errors.npy", DAMN_model_errors_GtoA)

## Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_iteration_for_concurrent function to evaluate the RL_data_in_GtoA data
start = time.time()
for i in range(RL_data_in_GtoA.shape[0]):
    start_i = time.time()
    
    #The kernel stays the same only for data with the same horizontal axis position
    kernel_GtoA = Get_PSF_kernel(PSF_width_value, RL_horizontal_axis_GtoA[i])
    
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_iteration_for_concurrent, kernel_GtoA)
        res = pool.map(intermediate_func, RL_data_in_GtoA[i])
    RL_output_GtoA[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_GtoA.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/Transition_RL_output.npy", RL_output_GtoA)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_errors_GtoA = Evaluate_metric(RL_output_GtoA, RL_data_target_GtoA)

np.save("Data_files/Processed_data/Transition_RL_errors.npy", RL_errors_GtoA)

## Total variation Richardson-Lucy

In [ ]:
#The ProcessPoolExecutor calling RL_TV_iteration_for_concurrent function to evaluate the RL_data_in_GtoA data
start = time.time()
for i in range(RL_data_in_GtoA.shape[0]):
    start_i = time.time()
    
    #The kernel stays the same only for data with the same horizontal axis position
    kernel_GtoA = Get_PSF_kernel(PSF_width_value, RL_horizontal_axis_GtoA[i])
    
    with concurrent.futures.ProcessPoolExecutor(CPU_units_to_use) as pool:
        intermediate_func = functools.partial(RL_TV_iteration_for_concurrent, kernel_GtoA)
        res = pool.map(intermediate_func, RL_data_in_GtoA[i])
    RL_TV_output_GtoA[i] = np.array(list(res))
    print("Finished iteration", i+1, "out of", RL_data_in_GtoA.shape[0], "in", np.round(time.time()-start_i, 2), 
          "seconds (" + str(np.round(time.time()-start, 2)), "from start).")

np.save("Data_files/Processed_data/Transition_RLTV_output.npy", RL_TV_output_GtoA)

In [ ]:
#Evaluate a chosen metric comparing with corresponding target images
RL_TV_errors_GtoA = Evaluate_metric(RL_TV_output_GtoA, RL_data_target_GtoA)

np.save("Data_files/Processed_data/Transition_RLTV_errors.npy", RL_TV_errors_GtoA)